# Customer Churn Prediction

**Problem:** Losing customers is expensive. If a company can predict which customers are likely to leave, the retention team can contact them early with offers or support.

**Goal:** Build a machine learning model that predicts whether a telecom customer will churn, and find the main factors behind churn.

**Dataset:** Telco Customer Churn (IBM sample dataset, 7,043 customers, 21 columns).

**Tools:** Python, Pandas, NumPy, Matplotlib, Seaborn, Scikit-learn

## 1. Import libraries and load data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

from sklearn.model_selection import train_test_split, cross_validate, StratifiedKFold
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report, RocCurveDisplay
)

sns.set_style("whitegrid")

# Automatically download the Telco Customer Churn dataset if it is not present
file_path = '/content/telco.csv'
if not os.path.exists(file_path):
    print("Dataset not found locally. Downloading Telco Customer Churn dataset...")
    url = 'https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv'
    df = pd.read_csv(url)
    df.to_csv(file_path, index=False)
else:
    df = pd.read_csv(file_path)

print("Shape:", df.shape)
df.head()

## 2. Data cleaning

In [ ]:
df.info()

In [ ]:
# TotalCharges is stored as text because some rows contain blank spaces
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
print("Missing values:")
print(df.isnull().sum()[df.isnull().sum() > 0])
print("\nDuplicate rows:", df.duplicated().sum())

In [ ]:
# Blank TotalCharges belong to brand new customers (tenure = 0), so fill with 0
print(df[df["TotalCharges"].isnull()][["tenure", "MonthlyCharges", "TotalCharges"]].head())
df["TotalCharges"] = df["TotalCharges"].fillna(0)

# customerID is only an identifier, so drop it
df = df.drop(columns=["customerID"])

# Convert target to 0/1
df["Churn"] = df["Churn"].map({"No": 0, "Yes": 1})
df.shape

In [ ]:
# Data quality check: TotalCharges was the only column with blank values
raw = pd.read_csv(file_path)
blank = raw["TotalCharges"].astype(str).str.strip() == ""
print("Blank TotalCharges values:", blank.sum())
print("Tenure of those customers:", raw.loc[blank, "tenure"].unique().tolist())
print("Blank values in any other column:", int((raw.drop(columns="TotalCharges").astype(str).apply(lambda c: c.str.strip()) == "").sum().sum()))
print("Rows before and after cleaning:", len(raw), len(df))

## 3. Exploratory data analysis (EDA)

In [ ]:
churn_rate = df["Churn"].mean() * 100
print(f"Overall churn rate: {churn_rate:.1f}%")

fig, ax = plt.subplots(figsize=(5, 4))
df["Churn"].value_counts().rename({0: "Stayed", 1: "Churned"}).plot(kind="bar", color=["#4C9F70", "#D9534F"], ax=ax)
ax.set_title("Customers who stayed vs churned")
ax.set_ylabel("Customers")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

The data is imbalanced: about 26% of customers churned. Because of this, accuracy alone can mislead, so recall, F1-score and ROC-AUC are used as well.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
for ax, col in zip(axes, ["Contract", "InternetService", "PaymentMethod"]):
    rate = df.groupby(col)["Churn"].mean().sort_values(ascending=False) * 100
    sns.barplot(x=rate.values, y=rate.index, ax=ax, color="#4C78A8")
    ax.set_title(f"Churn rate by {col}")
    ax.set_xlabel("Churn rate (%)")
    ax.set_ylabel("")
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
sns.boxplot(data=df, x="Churn", y="tenure", hue="Churn", legend=False, ax=axes[0], palette=["#4C9F70", "#D9534F"])
axes[0].set_title("Tenure (months) vs churn")
axes[0].set_xticks([0, 1]); axes[0].set_xticklabels(["Stayed", "Churned"])
sns.boxplot(data=df, x="Churn", y="MonthlyCharges", hue="Churn", legend=False, ax=axes[1], palette=["#4C9F70", "#D9534F"])
axes[1].set_title("Monthly charges vs churn")
axes[1].set_xticks([0, 1]); axes[1].set_xticklabels(["Stayed", "Churned"])
plt.tight_layout()
plt.show()

In [ ]:
print(df.groupby("Contract")["Churn"].mean().mul(100).round(1))
print()
print(df.groupby("Churn")[["tenure", "MonthlyCharges"]].median())

## 4. Feature engineering and preprocessing

Two simple features are added: average charges per month and the number of selected services. Categorical columns are one-hot encoded, and the data is split into training and test sets using stratification.

**Leakage prevention:** scaling is performed inside the Logistic Regression pipeline. This means the scaler is fitted separately within each cross-validation training fold and never learns from that fold's validation data or from the held-out test set. Random Forest uses the unscaled encoded features.

In [ ]:
# Simple new features
df["AvgChargesPerMonth"] = np.where(
    df["tenure"] > 0,
    df["TotalCharges"] / df["tenure"],
    df["MonthlyCharges"]
)

service_cols = [
    "OnlineSecurity", "OnlineBackup", "DeviceProtection",
    "TechSupport", "StreamingTV", "StreamingMovies"
]
df["NumServices"] = (df[service_cols] == "Yes").sum(axis=1)

# One-hot encode categorical columns
X = pd.get_dummies(df.drop(columns=["Churn"]), drop_first=True)
y = df["Churn"]

print("Features after encoding:", X.shape[1])

# Stratified split preserves the churn / non-churn proportion in both sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Numeric columns are scaled inside a Pipeline, so each CV fold learns
# its own scaling parameters from that fold's training portion only.
num_cols = [
    "tenure", "MonthlyCharges", "TotalCharges",
    "AvgChargesPerMonth", "NumServices"
]

print("Train:", X_train.shape, "Test:", X_test.shape)
print("Test set size:", len(X_test))
print("Training churn rate:", round(y_train.mean() * 100, 2), "%")
print("Test churn rate:", round(y_test.mean() * 100, 2), "%")

## 5. Model training

In [ ]:
# Use the same shuffled, stratified folds for both models.
cv_strategy = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Scale only numeric columns for Logistic Regression.
# The scaler is inside the Pipeline to prevent cross-validation leakage.
lr_preprocessor = ColumnTransformer(
    transformers=[("scale_numeric", StandardScaler(), num_cols)],
    remainder="passthrough",
    verbose_feature_names_out=False
)

models = {
    "Logistic Regression": Pipeline([
        ("preprocessing", lr_preprocessor),
        ("model", LogisticRegression(
            max_iter=1000, class_weight="balanced", random_state=42
        ))
    ]),
    "Random Forest": RandomForestClassifier(
        n_estimators=300,
        max_depth=8,
        min_samples_leaf=5,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    )
}

results = []
fitted = {}

scoring = {
    "roc_auc": "roc_auc",
    "recall": "recall"
}

for name, model in models.items():
    # Cross-validation is performed on training data only.
    cv_output = cross_validate(
        model,
        X_train,
        y_train,
        cv=cv_strategy,
        scoring=scoring,
        n_jobs=-1,
        return_train_score=False
    )

    # Fit once on the full training set, then evaluate on the untouched test set.
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    proba = model.predict_proba(X_test)[:, 1]

    fitted[name] = (model, X_test, pred, proba)

    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, pred),
        "Precision": precision_score(y_test, pred, zero_division=0),
        "Recall": recall_score(y_test, pred, zero_division=0),
        "F1-score": f1_score(y_test, pred, zero_division=0),
        "ROC-AUC": roc_auc_score(y_test, proba),
        "CV ROC-AUC mean": cv_output["test_roc_auc"].mean(),
        "CV ROC-AUC std": cv_output["test_roc_auc"].std(),
        "CV Recall mean": cv_output["test_recall"].mean(),
        "CV Recall std": cv_output["test_recall"].std()
    })

results_df = pd.DataFrame(results).set_index("Model")
display(results_df.round(3))

print("Note: CV scores are calculated on the training set only.")
print("Test-set scores are calculated once on the separate 20% holdout set.")

## 6. Model evaluation

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
for ax, (name, (model, Xte, pred, proba)) in zip(axes, fitted.items()):
    sns.heatmap(confusion_matrix(y_test, pred), annot=True, fmt="d", cmap="Blues", ax=ax,
                xticklabels=["Stayed", "Churned"], yticklabels=["Stayed", "Churned"])
    ax.set_title(f"{name}\nConfusion matrix")
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual")
plt.tight_layout()
plt.show()

### Final confusion matrix for the recommended model (saved as an image for the portfolio)

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay
from matplotlib.colors import LinearSegmentedColormap

lr_pipeline = models["Logistic Regression"]

cmap = LinearSegmentedColormap.from_list("site", ["#FAFAF7", "#1F3A5F"])
fig, ax = plt.subplots(figsize=(5, 4))
disp = ConfusionMatrixDisplay.from_predictions(
    y_test, lr_pipeline.predict(X_test), display_labels=["Stayed", "Left"],
    cmap=cmap, colorbar=False, ax=ax
)
ax.grid(False)
ax.set_title("Churn model on 1,409 test customers", fontsize=11)

# white text on dark cells, dark text on light cells
for text in disp.text_.ravel():
    value = int(text.get_text())
    text.set_color("white" if value > 400 else "#14181F")
    text.set_fontsize(13)

fig.savefig("confusion_matrix.png", dpi=200, bbox_inches="tight", facecolor="#FAFAF7")
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
from sklearn.metrics import roc_curve
roc_colors = {"Logistic Regression": "#1F3A5F", "Random Forest": "#9CA3AF"}
for name, (model, Xte, pred, proba) in fitted.items():
    fpr, tpr, _ = roc_curve(y_test, proba)
    ax.plot(fpr, tpr, color=roc_colors[name], linewidth=2,
            label=f"{name} (AUC = {roc_auc_score(y_test, proba):.3f})")
ax.plot([0, 1], [0, 1], linestyle="--", color="#4B5563", label="Random guess")
ax.set_xlabel("False positive rate")
ax.set_ylabel("True positive rate")
ax.grid(False)
ax.set_title("ROC curves on 1,409 test customers", fontsize=11)
ax.legend()
plt.tight_layout()
fig.savefig("roc_curves.png", dpi=200, bbox_inches="tight", facecolor="#FAFAF7")
plt.show()

## 7. What drives churn? (feature importance)

In [ ]:
rf = fitted["Random Forest"][0]
imp = pd.Series(rf.feature_importances_, index=X_train.columns).sort_values(ascending=False).head(10)

fig, ax = plt.subplots(figsize=(8, 5))
sns.barplot(x=imp.values, y=imp.index, color="#4C78A8", ax=ax)
ax.set_title("Top 10 features (Random Forest)")
ax.set_xlabel("Importance")
plt.tight_layout()
plt.show()

In [ ]:
# Logistic Regression coefficients from the fitted Pipeline
lr_pipeline = fitted["Logistic Regression"][0]
lr_model = lr_pipeline.named_steps["model"]
feature_names = lr_pipeline.named_steps["preprocessing"].get_feature_names_out()

coef = pd.Series(lr_model.coef_[0], index=feature_names).sort_values()

print("Features associated with LOWER churn (negative coefficients):")
print(coef.head(5).round(2))
print("\nFeatures associated with HIGHER churn (positive coefficients):")
print(coef.tail(5).round(2))
print("\nInterpretation: coefficients describe associations in this fitted model;")
print("they do not prove that a feature causes a customer to churn.")

In [ ]:
# Chart of the clearest coefficients (the charge columns overlap with tenure, so they are left out)
c = coef.drop(["MonthlyCharges", "TotalCharges", "AvgChargesPerMonth"], errors="ignore")
top = pd.concat([c.head(5), c.tail(5)])

fig, ax = plt.subplots(figsize=(7, 4.5))
ax.barh(top.index, top.values, color="#1F3A5F")
ax.axvline(0, color="#4B5563", linewidth=0.8)
ax.grid(False)
ax.set_title("What moves churn in the Logistic Regression model", fontsize=11)
ax.set_xlabel("Coefficient (negative = lower churn, positive = higher)")
plt.tight_layout()
fig.savefig("coefficients.png", dpi=200, bbox_inches="tight", facecolor="#FAFAF7")
plt.show()

## 8. Model comparison summary

In [ ]:
# Compare models on the independent test set by ROC-AUC.
best = results_df["ROC-AUC"].idxmax()
print("Highest test ROC-AUC:", best)
print(results_df.round(3).to_string())

print("\nConfusion-matrix counts (actual rows: stayed, churned; predicted columns: stayed, churned):")
for name, (_, _, pred, _) in fitted.items():
    cm = confusion_matrix(y_test, pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()
    print(f"\n{name}")
    print(f"TN={tn}, FP={fp}, FN={fn}, TP={tp}")
    print(f"Actual churners detected: {tp} out of {tp + fn}")

print("\nChurn rate by contract type (%):")
print(df.groupby("Contract")["Churn"].mean().mul(100).round(1).to_string())

## 9. Conclusion and recommendation

**Results on 1,409 held-out test customers.**

| Model | Accuracy | Recall | Precision | ROC-AUC |
|---|---|---|---|---|
| Logistic Regression | 74.0% | 78.6% | 50.6% | 0.842 |
| Random Forest | 76.1% | 77.8% | 53.4% | 0.844 |
| Always predict "no churn" | 73.5% | 0% | n/a | n/a |

The ROC-AUC gap between the two models (0.002) is much smaller than the spread across cross-validation folds (about 0.012), so the models are effectively tied on that measure.

**Why recall.** A model that always predicts "no churn" scores 73.5% accuracy and catches no churners. Logistic Regression catches 294 of the 374 customers who left (Random Forest: 291).

**Recommendation: Logistic Regression.** Its coefficients are easy to read, and its recall is as good or better than Random Forest (294 vs 291 churners caught on the test set).

**What is linked to churn.** Month-to-month contracts churn at 42.7%, against 11.3% for one-year and 2.8% for two-year contracts. In the model, two-year contracts, one-year contracts and longer tenure go with lower churn, and fiber optic internet goes with higher churn. These are associations, not proven causes.

**Business use.** A retention team can use the flagged list to contact likely churners first, starting with month-to-month customers.

**Limits.**
- About half of the customers the model flags stay (287 of 581), so contacting them has a cost.
- I assume a missed churner costs more than a wasted contact. I have not tested that cost.
- I have not tested whether contacting flagged customers keeps them.
- The 0.5 decision threshold and the model settings were not tuned.
- Results come from a single 80/20 split, so numbers could shift a little with another split.
- The data is the public IBM Telco sample, not a company's own customers.